# agentorch 实验手册（完整功能版）

这个 notebook 基于当前最新的 `agentorch` 代码重写，目标是把**当前已经支持的全部核心能力**都整理成可以直接实验的示例。

你可以把它当成：

1. 快速上手手册
2. 功能验证脚本
3. 架构设计示例集

当前覆盖能力包括：

- OpenAI-compatible 模型接入
- 单 agent 基础运行
- 结构化工具注册与调用
- 代码解释器与 sandbox 执行
- memory 与 thread 清理
- workflow 基础编排
- RAG-ready 检索接口与最小本地实现
- AgentRegistry 注册中心
- supervisor 多智能体动态委派
- workflow 中的 agent 节点


## 0. 先看最重要的使用规则

在 Jupyter / Notebook 里，请记住这几条：

- 不要调用 `agent.run_sync(...)`
- 必须使用 `await agent.run(...)`
- 修改了本地包代码之后，要重启 kernel 再重新运行
- tool calling 或多 agent 调试时，建议使用新的 `thread_id`
- 如果想复用旧 `thread_id`，先执行 `await runtime.memory.clear_thread(...)`

这是因为 notebook 会缓存对象和内存状态，旧状态很容易影响新实验。


## 1. 环境准备

推荐 Python 版本：`3.11+`

当前项目建议使用 `py -3.13`。

推荐安装命令：

```powershell
py -3.13 -m pip install -e .
py -3.13 -m pip install jupyterlab notebook
```


In [1]:
import sys
from pathlib import Path

print(sys.version)
print(Path.cwd())


3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:39:58) [MSC v.1943 64 bit (AMD64)]
c:\Users\24260\Desktop\研究生生涯\智能体开发范式


## 2. API Key / Base URL 导入方式

`agentorch` 会自动读取项目根目录下的 `.env`，并兼容两套变量名：

- `OPENAI_API_KEY` / `OPENAI_BASE_URL`
- `API_KEY` / `BASE_URL`

推荐写法：

```env
OPENAI_API_KEY=sk-xxxx
OPENAI_BASE_URL=https://api.openai.com/v1
```

如果你用 OpenAI-compatible 代理，也支持：

```env
API_KEY=sk-xxxx
BASE_URL=https://your-proxy.example.com/v1/chat/completions
```

框架会自动把 `.../chat/completions` 规整为 SDK 所需的 `/v1` 根地址。


In [2]:
from agentorch.config.settings import ModelConfig

cfg = ModelConfig()
print('api_key_loaded:', bool(cfg.api_key))
print('base_url:', cfg.base_url)
print('default_model:', cfg.model)


api_key_loaded: True
base_url: https://www.dmxapi.cn/v1
default_model: gpt-4.1-mini


## 3. 当前公开 API


In [3]:
import agentorch

print(agentorch.__all__)


['Agent', 'AgentRegistry', 'AgentSpec', 'BaseRetriever', 'Context', 'InMemoryKnowledgeBase', 'KnowledgeBase', 'MemoryManager', 'OpenAIModel', 'Runtime', 'SandboxManager', 'SkillLoader', 'SkillRegistry', 'Supervisor', 'TaskPacket', 'ToolRegistry', 'Workflow', 'create_python_interpreter_tool', 'tool']


## 4. 最小 Agent 运行

这是最基础的一条链路：

- `OpenAIModel` 负责模型接入
- `Runtime` 负责组织运行时
- `Agent` 提供开发者直接调用接口

注意：**在 notebook 中必须使用 `await agent.run(...)`。**


In [4]:
from agentorch import Agent, OpenAIModel, Runtime

model = OpenAIModel(model='gpt-4.1')
runtime = Runtime(model=model)
agent = Agent(runtime=runtime)

result = await agent.run(
    '请用三句话介绍 agentorch 这个包的作用。',
    thread_id='nb-basic-001',
)

print(result.output_text)
print(result.usage)


{"event_type": "run_started", "request_id": "4b72ae60-1f57-4ea4-b53a-97e7e4258a6c", "run_id": "43e8cbda-de1a-406e-9ee8-1d04818a58c2", "thread_id": "nb-basic-001", "trace_id": "3429c211-2386-4357-925e-dcaed37664ba", "metadata": {}}
{"event_type": "prompt_built", "request_id": "4b72ae60-1f57-4ea4-b53a-97e7e4258a6c", "run_id": "43e8cbda-de1a-406e-9ee8-1d04818a58c2", "thread_id": "nb-basic-001", "trace_id": "3429c211-2386-4357-925e-dcaed37664ba", "metadata": {}, "step": 0, "message_count": 2}
{"event_type": "model_called", "request_id": "4b72ae60-1f57-4ea4-b53a-97e7e4258a6c", "run_id": "43e8cbda-de1a-406e-9ee8-1d04818a58c2", "thread_id": "nb-basic-001", "trace_id": "3429c211-2386-4357-925e-dcaed37664ba", "metadata": {}, "step": 0, "finish_reason": "stop", "tool_calls": 0}
{"event_type": "memory_written", "request_id": "4b72ae60-1f57-4ea4-b53a-97e7e4258a6c", "run_id": "43e8cbda-de1a-406e-9ee8-1d04818a58c2", "thread_id": "nb-basic-001", "trace_id": "3429c211-2386-4357-925e-dcaed37664ba", "me

agentoгch 是一个基于 PyTorch 的强化学习和元学习研究框架。它提供了灵活的环境、策略和实验管理工具，便于快速实现和测试新算法。该框架支持模块化设计，方便用户定制和扩展各类智能体模型。
prompt_tokens=67 completion_tokens=65 total_tokens=132 estimated_cost=None


## 5. 结构化工具实验

下面演示如何定义一个工具、注册到 `ToolRegistry`、再由模型自动决定是否调用。


In [5]:
from pydantic import BaseModel
from agentorch import Agent, OpenAIModel, Runtime, ToolRegistry, tool


class AddInput(BaseModel):
    a: int
    b: int


@tool(description='Add two integers together.')
async def add_numbers(input: AddInput):
    return {'sum': input.a + input.b}


tools = ToolRegistry()
tools.register(add_numbers)

runtime = Runtime(model=OpenAIModel(model='gpt-4.1'), tools=tools)
agent = Agent(runtime=runtime)

result = await agent.run(
    '请调用 add_numbers 工具，计算 123 + 456，并解释结果。',
    thread_id='nb-tool-003',
)

print(result.output_text)
print(result.tool_results)


{"event_type": "run_started", "request_id": "d3928ba0-4f35-469f-ab26-88e3d5248be6", "run_id": "11fb100b-5e8b-4bcd-84a5-5b6c2ecb0efc", "thread_id": "nb-tool-003", "trace_id": "5dc1840d-20a1-4c9e-be61-826eeff55c6d", "metadata": {}}
{"event_type": "prompt_built", "request_id": "d3928ba0-4f35-469f-ab26-88e3d5248be6", "run_id": "11fb100b-5e8b-4bcd-84a5-5b6c2ecb0efc", "thread_id": "nb-tool-003", "trace_id": "5dc1840d-20a1-4c9e-be61-826eeff55c6d", "metadata": {}, "step": 0, "message_count": 2}
{"event_type": "model_called", "request_id": "d3928ba0-4f35-469f-ab26-88e3d5248be6", "run_id": "11fb100b-5e8b-4bcd-84a5-5b6c2ecb0efc", "thread_id": "nb-tool-003", "trace_id": "5dc1840d-20a1-4c9e-be61-826eeff55c6d", "metadata": {}, "step": 0, "finish_reason": "tool_calls", "tool_calls": 1}
{"event_type": "tool_called", "request_id": "d3928ba0-4f35-469f-ab26-88e3d5248be6", "run_id": "11fb100b-5e8b-4bcd-84a5-5b6c2ecb0efc", "thread_id": "nb-tool-003", "trace_id": "5dc1840d-20a1-4c9e-be61-826eeff55c6d", "met

123 + 456 的计算结果是 579。

解释：这是两个整数相加的基本算术运算。将 123 和 456 相加，结果等于 579。
[ToolExecutionResult(tool_call_id='call_AaHBsAmEybKhVW8PvDGJ9J5r', tool_name='add_numbers', output={'sum': 579}, is_error=False, error_message=None, duration=0.00014179999925545417, metadata={})]


### 如果你想复用旧 thread_id

先清理旧线程消息，避免旧状态残留：


In [ ]:
# 示例：
# await runtime.memory.clear_thread('nb-tool-001')


## 6. 查看工具调用的结构化结果


In [6]:
for item in result.tool_results:
    print(item.model_dump())


{'tool_call_id': 'call_AaHBsAmEybKhVW8PvDGJ9J5r', 'tool_name': 'add_numbers', 'output': {'sum': 579}, 'is_error': False, 'error_message': None, 'duration': 0.00014179999925545417, 'metadata': {}}


## 7. 代码解释器实验

`agentorch` 里的“代码解释器”是通过 `python_interpreter` 工具 + `SandboxManager` 组合出来的，而不是写死在系统里的黑盒。


In [7]:
from pathlib import Path
from agentorch import Agent, OpenAIModel, Runtime, SandboxManager, ToolRegistry, create_python_interpreter_tool
from agentorch.sandbox import SandboxPolicy

sandbox = SandboxManager(
    policy=SandboxPolicy(
        allowed_paths=[Path.cwd()],
        command_allowlist=['python'],
        timeout=15.0,
    )
)

tools = ToolRegistry()
tools.register(create_python_interpreter_tool(sandbox))

runtime = Runtime(
    model=OpenAIModel(model='gpt-4.1'),
    tools=tools,
    sandbox=sandbox,
)
agent = Agent(runtime=runtime)

result = await agent.run(
    '请使用 python_interpreter 计算前 10 个斐波那契数，并简要解释结果。',
    thread_id='nb-code-002',
)

print(result.output_text)
for item in result.tool_results:
    print(item.model_dump())


{"event_type": "run_started", "request_id": "f4af36ae-5d4a-47b0-8ca4-98f6f53b5714", "run_id": "2ba74740-ea01-4a16-9b9e-6337164ee2eb", "thread_id": "nb-code-002", "trace_id": "6349f5c1-2945-4e8a-a193-179f36383e5b", "metadata": {}}
{"event_type": "prompt_built", "request_id": "f4af36ae-5d4a-47b0-8ca4-98f6f53b5714", "run_id": "2ba74740-ea01-4a16-9b9e-6337164ee2eb", "thread_id": "nb-code-002", "trace_id": "6349f5c1-2945-4e8a-a193-179f36383e5b", "metadata": {}, "step": 0, "message_count": 2}
{"event_type": "model_called", "request_id": "f4af36ae-5d4a-47b0-8ca4-98f6f53b5714", "run_id": "2ba74740-ea01-4a16-9b9e-6337164ee2eb", "thread_id": "nb-code-002", "trace_id": "6349f5c1-2945-4e8a-a193-179f36383e5b", "metadata": {}, "step": 0, "finish_reason": "tool_calls", "tool_calls": 1}
{"event_type": "tool_called", "request_id": "f4af36ae-5d4a-47b0-8ca4-98f6f53b5714", "run_id": "2ba74740-ea01-4a16-9b9e-6337164ee2eb", "thread_id": "nb-code-002", "trace_id": "6349f5c1-2945-4e8a-a193-179f36383e5b", "met

前 10 个斐波那契数是：[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]。

简要解释：
斐波那契数列的特点是，从第3项开始，每一项都等于前两项的和。首两项分别为0和1。这个序列经常出现在自然规律、算法和数学问题中。
{'tool_call_id': 'call_AXASSpy9Co3BOJh4RnfH2Fde', 'tool_name': 'python_interpreter', 'output': {'stdout': '[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\n', 'stderr': '', 'exit_code': 0, 'duration': 0.18514460000005784}, 'is_error': False, 'error_message': None, 'duration': 0.18753300000025774, 'metadata': {}}


## 8. 不通过 Agent，直接调用代码解释器工具


In [11]:
from pathlib import Path

from agentorch import SandboxManager, create_python_interpreter_tool
from agentorch.sandbox import SandboxPolicy

sandbox = SandboxManager(
    policy=SandboxPolicy(
        allowed_paths=[Path.cwd()],
        command_allowlist=["python"],
        timeout=10.0,
    )
)

tool = create_python_interpreter_tool(sandbox)

interpreter_result = await tool.run(
    tool.input_model(
        code="""
values = [1, 2, 3, 4]
print(sum(values))
print(values[-1])
""",
        workdir=str(Path.cwd()),
    )
)

interpreter_result.model_dump()


{'tool_name': 'python_interpreter',
 'data': {'stdout': '10\n4\n',
  'stderr': '',
  'exit_code': 0,
  'duration': 0.19654240000090795},
 'success': True,
 'error': None,
 'duration': 0.19788470000275993}

## 9. Memory 实验

当前 memory 主要支持：

- thread message
- thread summary
- long-term record
- checkpoint
- clear_thread


In [12]:
from agentorch.memory import MemoryManager, MemoryRecord
from agentorch.core import Message

memory = MemoryManager()

await memory.append_message('thread-demo', Message(role='user', content='我偏好异步优先架构'))
await memory.append_message('thread-demo', Message(role='assistant', content='收到，我会按异步优先来设计。'))

summary = await memory.summarize_thread('thread-demo')
print(summary)

await memory.remember(
    MemoryRecord(
        thread_id='thread-demo',
        kind='preference',
        content='用户偏好异步优先架构',
        tags=['preference', 'architecture'],
    )
)

records = await memory.search(thread_id='thread-demo', query='异步')
records


user: 我偏好异步优先架构
assistant: 收到，我会按异步优先来设计。


[{'id': 15,
  'thread_id': 'thread-demo',
  'kind': 'preference',
  'content': '用户偏好异步优先架构',
  'tags': ['preference', 'architecture']},
 {'id': 25,
  'thread_id': 'thread-demo',
  'kind': 'preference',
  'content': '用户偏好异步优先架构',
  'tags': ['preference', 'architecture']}]

## 10. 清理线程上下文


In [13]:
await memory.clear_thread('thread-demo')
print(await memory.get_thread_messages('thread-demo'))


[]


## 11. Workflow 基础实验

当前 workflow 支持这些节点类型：

- `model`
- `tool`
- `router`
- `memory`
- `agent`

先从最基础的 memory + model 流程开始。


In [14]:
from agentorch import OpenAIModel, Runtime
from agentorch.runtime import Agent
from agentorch.workflow import Edge, Node, Workflow

workflow = Workflow(
    entry_node='remember_project',
    nodes=[
        Node(
            id='remember_project',
            kind='memory',
            config={
                'action': 'remember',
                'kind': 'project_note',
                'content': 'agentorch 是一个代码优先、异步优先的智能体编排框架。',
                'tags': ['project', 'overview'],
            },
        ),
        Node(
            id='summarize',
            kind='model',
            config={
                'prompt': '请总结 agentorch 的定位和适合的使用场景。',
            },
        ),
    ],
    edges=[
        Edge(source='remember_project', target='summarize', kind='success'),
    ],
)

runtime = Runtime(model=OpenAIModel(model='gpt-4.1'))
agent = Agent(runtime=runtime, workflow=workflow)

result = await agent.run('请执行 workflow', thread_id='nb-workflow-001')
print(result.output_text)


{"event_type": "run_started", "request_id": "a3c416c8-9d75-4eca-84d2-b73ceba1681a", "run_id": "7f62bff9-dc11-4a7e-93da-838d4c3a3ed7", "thread_id": "nb-workflow-001", "trace_id": "fc029d26-f8ec-46b4-a0ad-a90e823e0dc7", "metadata": {}}
{"event_type": "run_started", "request_id": "ad802818-454e-4615-aac6-f31a317a9e5d", "run_id": "85db2cd1-965c-4c3b-8fe5-824fd9c3bbd6", "thread_id": "nb-workflow-001", "trace_id": "52cb0a5a-a84a-449d-92d4-2fe99ae36e99", "metadata": {}}
{"event_type": "prompt_built", "request_id": "ad802818-454e-4615-aac6-f31a317a9e5d", "run_id": "85db2cd1-965c-4c3b-8fe5-824fd9c3bbd6", "thread_id": "nb-workflow-001", "trace_id": "52cb0a5a-a84a-449d-92d4-2fe99ae36e99", "metadata": {}, "step": 0, "message_count": 3}
{"event_type": "model_called", "request_id": "ad802818-454e-4615-aac6-f31a317a9e5d", "run_id": "85db2cd1-965c-4c3b-8fe5-824fd9c3bbd6", "thread_id": "nb-workflow-001", "trace_id": "52cb0a5a-a84a-449d-92d4-2fe99ae36e99", "metadata": {}, "step": 0, "finish_reason": "st

{"status": "completed", "output_text": "agento​​rch 的定位  \nagento​​rch 是一个面向多智能体系统（multi-agent systems, MAS）和基于学习的智能体开发的 Python 框架。它秉承灵活、高度可扩展的设计理念，主要面向研究人员和开发者，致力于简化和标准化复杂环境下多智能体的开发、训练和评估流程。它集成了 PyTorch，支持神经网络模型、分布式训练及强化学习（RL）算法，允许用户高效实现多种多智能体交互机制与训练范式。\n\n适合的使用场景  \n\n1. 多智能体强化学习（Multi-Agent Reinforcement Learning, MARL）  \n最典型的应用场景，适用于并发、多智能体协作或对抗的问题，如团队游戏、机器人协作等。agento​​rch 提供了必要的基础设施与算法支持，如 MADDPG、MAPPO 等。\n2. 多智能体环境仿真与评测  \n适合需要自定义、模拟或对比不同智能体策略表现的实验环境（如 GridWorld、StarCraft II、交通系统模拟等）。\n3. 智能体交互机制研究  \n开发和实验通信、博弈论、竞争与协作等多智能体系统中的创新算法与协议。\n4. 深度学习与分布式系统结合  \n当需要快速搭建大规模、分布式多智能体实验时，agento​​rch 整合了分布式训练与资源管理模块。\n5. 学术研究与教学  \n适合算法原型验证、论文复现以及研究型课程项目。\n\n总结  \nagento​​rch 定位于多智能体智能系统领域，是专为高效实现、扩展和评测多智能体学习算法而设计的强大工具，尤其适合需快速迭代和高可定制化的科研或工程项目。"}


## 12. RAG-ready 接口层实验

当前版本已经支持：

- `Document`
- `DocumentChunk`
- `RetrievalQuery`
- `RetrievedChunk`
- `BaseRetriever`
- `KnowledgeBase`
- `RAGContextBuilder`
- 最小本地实现 `InMemoryKnowledgeBase`

这不是完整生产级 RAG，而是“标准接口 + 最小可运行实现”。


In [15]:
from agentorch import InMemoryKnowledgeBase
from agentorch.knowledge import Document, RetrievalQuery, RAGContextBuilder

knowledge_base = InMemoryKnowledgeBase()
await knowledge_base.ingest(
    [
        Document(id='doc-1', text='agentorch is a code-first, async-first agent orchestration framework for Python.'),
        Document(id='doc-2', text='RAG augments prompts with retrieved knowledge.'),
    ]
)

retriever = knowledge_base.get_retriever()
chunks = await retriever.retrieve(RetrievalQuery(query='agentorch framework', top_k=3))

print(chunks)
print('--- RAG context ---')
print(RAGContextBuilder().build(chunks))


[RetrievedChunk(chunk=DocumentChunk(id='doc-1-chunk-1', document_id='doc-1', text='agentorch is a code-first, async-first agent orchestration framework for Python.', metadata={}), score=2.0, source='keyword')]
--- RAG context ---
[1] score=2.000 doc=doc-1 text=agentorch is a code-first, async-first agent orchestration framework for Python.


## 13. 把检索接入 Runtime


In [16]:
from agentorch import Agent, OpenAIModel, Runtime, InMemoryKnowledgeBase
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document

knowledge_base = InMemoryKnowledgeBase()
await knowledge_base.ingest(
    [
        Document(
            id='doc-1',
            text='agentorch is designed for code-first, async-first agent orchestration in Python.'
        )
    ]
)

runtime = Runtime(
    model=OpenAIModel(model='gpt-4.1'),
    knowledge_base=knowledge_base,
    config=RuntimeConfig(enable_retrieval=True, max_retrieved_chunks=3),
)
agent = Agent(runtime=runtime)

result = await agent.run(
    'What is agentorch designed for?',
    thread_id='nb-rag-001',
)

print(result.output_text)


{"event_type": "run_started", "request_id": "e610dcd5-ff91-416d-a185-8d5d75f66527", "run_id": "b77c0fce-32a0-4640-8a5d-9570489c305e", "thread_id": "nb-rag-001", "trace_id": "194b33cd-6557-4409-aba1-b9a190629209", "metadata": {}}
{"event_type": "retrieval_started", "request_id": "e610dcd5-ff91-416d-a185-8d5d75f66527", "run_id": "b77c0fce-32a0-4640-8a5d-9570489c305e", "thread_id": "nb-rag-001", "trace_id": "194b33cd-6557-4409-aba1-b9a190629209", "metadata": {}, "query": "What is agentorch designed for?"}
{"event_type": "retrieval_completed", "request_id": "e610dcd5-ff91-416d-a185-8d5d75f66527", "run_id": "b77c0fce-32a0-4640-8a5d-9570489c305e", "thread_id": "nb-rag-001", "trace_id": "194b33cd-6557-4409-aba1-b9a190629209", "metadata": {}, "query": "What is agentorch designed for?", "chunk_count": 1}
{"event_type": "prompt_built", "request_id": "e610dcd5-ff91-416d-a185-8d5d75f66527", "run_id": "b77c0fce-32a0-4640-8a5d-9570489c305e", "thread_id": "nb-rag-001", "trace_id": "194b33cd-6557-4409

agentorch is designed for code-first, async-first agent orchestration in Python.


## 14. AgentRegistry 实验

多智能体系统里，agent 需要先注册，再被 workflow 或 supervisor 调度。


In [19]:
from agentorch import AgentRegistry, AgentSpec

registry = AgentRegistry()
print(registry.list_specs())


[]


## 15. 构建一个 specialist agent 并注册

下面构建一个最小 specialist agent，并注册到 `AgentRegistry`。


In [21]:
from pydantic import BaseModel
from agentorch import Agent, AgentRegistry, AgentSpec, OpenAIModel, Runtime, ToolRegistry, tool


class EchoInput(BaseModel):
    text: str


@tool(description='Echo a message as structured data.')
async def echo(input: EchoInput):
    return {'echo': input.text}


def build_specialist_agent(description: str) -> Agent:
    tools = ToolRegistry()
    tools.register(echo)
    runtime = Runtime(model=OpenAIModel(model='gpt-4.1'), tools=tools)
    return Agent(runtime=runtime)


registry = AgentRegistry()
planner_agent = build_specialist_agent('Planning specialist for decomposition tasks')
registry.register(
    AgentSpec(
        name='planner',
        description='Planning specialist',
        tags=['plan', 'task', 'workflow'],
    ),
    planner_agent,
)

print([spec.model_dump() for spec in registry.list_specs()])


[{'name': 'planner', 'description': 'Planning specialist', 'tags': ['plan', 'task', 'workflow'], 'input_schema': {}, 'output_schema': {}, 'metadata': {}}]


## 16. Supervisor 多智能体动态委派实验


In [22]:
from agentorch import Agent, OpenAIModel, Runtime, Supervisor

supervisor = Supervisor(registry=registry)
runtime = Runtime(
    model=OpenAIModel(model='gpt-4.1'),
    agent_registry=registry,
    supervisor=supervisor,
)
agent = Agent(runtime=runtime)

result = await agent.run(
    'Please plan the implementation steps for a Python agent framework.',
    thread_id='nb-supervisor-001',
)

print(result.output_text)


{"event_type": "run_started", "request_id": "31d858f9-d1fa-4315-aafc-d2dfb3951d8d", "run_id": "92755988-f092-4298-95b3-4414428dbc82", "thread_id": "nb-supervisor-001", "trace_id": "5b499fe7-cdc4-421f-82c1-02d00a83387a", "metadata": {}}
{"event_type": "supervisor_routed", "request_id": "31d858f9-d1fa-4315-aafc-d2dfb3951d8d", "run_id": "92755988-f092-4298-95b3-4414428dbc82", "thread_id": "nb-supervisor-001", "trace_id": "5b499fe7-cdc4-421f-82c1-02d00a83387a", "metadata": {}, "task_id": "92755988-f092-4298-95b3-4414428dbc82"}


{"event_type": "handoff_created", "request_id": "31d858f9-d1fa-4315-aafc-d2dfb3951d8d", "run_id": "92755988-f092-4298-95b3-4414428dbc82", "thread_id": "nb-supervisor-001", "trace_id": "5b499fe7-cdc4-421f-82c1-02d00a83387a", "metadata": {}, "agent_name": "planner", "task_id": "92755988-f092-4298-95b3-4414428dbc82:planner"}
{"event_type": "agent_delegated", "request_id": "31d858f9-d1fa-4315-aafc-d2dfb3951d8d", "run_id": "92755988-f092-4298-95b3-4414428dbc82", "thread_id": "nb-supervisor-001", "trace_id": "5b499fe7-cdc4-421f-82c1-02d00a83387a", "metadata": {}, "agent_name": "planner", "task_id": "92755988-f092-4298-95b3-4414428dbc82:planner"}
{"event_type": "run_started", "request_id": "1ef8f605-0ef5-4857-9889-5449173e57d4", "run_id": "0cb36016-35f4-467a-8b41-7a0629ab4a3c", "thread_id": "nb-supervisor-001", "trace_id": "629a2eb5-63d5-4469-ae24-3685cc3e0b61", "metadata": {"task_packet": {"task_id": "92755988-f092-4298-95b3-4414428dbc82:planner", "goal": "Please plan the implementation step

[planner] Here is a structured plan for implementing a Python agent framework:

1. Requirements Gathering
   - Define core features: agent actions, communication, environment interaction, extensibility.
   - Identify target use cases (automation, task delegation, etc.).

2. Architecture Design
   - Choose framework structure (object-oriented, modular).
   - Design agent lifecycle: initialization, execution, termination.
   - Plan for plugin/tool integration and message passing.

3. Environment Setup
   - Select Python version and libraries (asyncio, threading, etc.).
   - Setup project repository (with README and basic folder structure).

4. Core Components Implementation
   - Agent base class: properties, methods, state management.
   - Agent manager/controller: orchestration, registration, monitoring.
   - Communication system: message passing, event handling.
   - Tool/plugin system: interface and loading mechanism.

5. Extensibility Features
   - Add support for custom agent types.

## 17. Workflow 中的 agent 节点实验

这条链路演示静态 DAG 里的 agent 节点委派。


In [23]:
from agentorch import OpenAIModel, Runtime
from agentorch.runtime import Agent
from agentorch.workflow import Node, Workflow

runtime = Runtime(model=OpenAIModel(model='gpt-4.1'), agent_registry=registry)
workflow = Workflow(
    entry_node='delegate',
    nodes=[
        Node(
            id='delegate',
            kind='agent',
            config={
                'agent_name': 'planner',
                'goal': 'Please plan the steps for building an async Python agent runtime.',
                'output_key': 'planner_output',
            },
        )
    ],
    edges=[],
)
agent = Agent(runtime=runtime, workflow=workflow)

result = await agent.run('ignored by workflow node config', thread_id='nb-workflow-agent-001')
print(result.output_text)


{"event_type": "run_started", "request_id": "202816e0-ca86-4381-b416-9d6b2ee6bd8a", "run_id": "25a4e10d-6df9-4f62-a12f-8a899fe42b4f", "thread_id": "nb-workflow-agent-001", "trace_id": "f1e36446-7c46-4dfa-891a-632de2ed4cb6", "metadata": {}}
{"event_type": "agent_delegated", "thread_id": "nb-workflow-agent-001", "agent_name": "planner", "task_id": "nb-workflow-agent-001:delegate", "node_id": "delegate"}
{"event_type": "run_started", "request_id": "dfb36e74-e050-43c9-ba61-3042f9bce968", "run_id": "58b65b47-f0a6-4662-b51d-ec05755733dc", "thread_id": "nb-workflow-agent-001", "trace_id": "cb3e858d-2cce-466a-a86a-c6a6cbcf0c0b", "metadata": {"task_packet": {"task_id": "nb-workflow-agent-001:delegate", "goal": "Please plan the steps for building an async Python agent runtime.", "input": {}, "context": {"workflow_node": "delegate", "variables": {}}, "constraints": [], "expected_output": null, "artifacts": [], "metadata": {"thread_id": "nb-workflow-agent-001", "delegation_depth": 1}}, "_delegated

{"status": "completed", "output_text": "Here’s a concise plan for building an async Python agent runtime:\n\n1. Requirements Analysis\n   - Define agent behaviors, communication protocols, and use cases.\n   - Identify target Python versions and async frameworks (e.g., asyncio, trio).\n\n2. Architecture Design\n   - Design core runtime structure: agent lifecycle, message handling, task queue.\n   - Plan modular components: agent manager, task dispatcher, plugin interface.\n\n3. Environment Setup\n   - Set up project structure, virtual environment, and dependency management.\n   - Choose async libraries and development tools.\n\n4. Core Implementation\n   - Implement async event loop and task scheduling.\n   - Develop agent class with async lifecycle methods (start/stop/receive/process).\n   - Add message passing (with async channels or queues).\n\n5. Plugin & Extensibility System\n   - Define plugin architecture for agent skills/actions.\n   - Provide async plugin loading/unloading.\n\

## 18. 当前完整装配模板

这一节展示当前版本比较完整的装配方式：模型 + 工具 + sandbox + 检索 + registry + supervisor。


In [25]:
from pathlib import Path
from agentorch import (
    Agent,
    AgentRegistry,
    OpenAIModel,
    Runtime,
    SandboxManager,
    Supervisor,
    ToolRegistry,
    InMemoryKnowledgeBase,
    create_python_interpreter_tool,
)
from agentorch.config import RuntimeConfig
from agentorch.knowledge import Document
from agentorch.sandbox import SandboxPolicy

sandbox = SandboxManager(
    policy=SandboxPolicy(
        allowed_paths=[Path.cwd()],
        command_allowlist=['python'],
        timeout=15.0,
    )
)

tools = ToolRegistry()
tools.register(create_python_interpreter_tool(sandbox))

knowledge_base = InMemoryKnowledgeBase()
await knowledge_base.ingest([
    Document(id='doc-1', text='agentorch combines runtime, tools, memory, workflow, retrieval, and multi-agent orchestration.')
])

registry = AgentRegistry()
supervisor = Supervisor(registry=registry)

runtime = Runtime(
    model=OpenAIModel(model='gpt-4.1'),
    tools=tools,
    sandbox=sandbox,
    knowledge_base=knowledge_base,
    agent_registry=registry,
    supervisor=supervisor,
    config=RuntimeConfig(enable_retrieval=True, max_retrieved_chunks=3),
)

agent = Agent(runtime=runtime)
print(agent)


## 19. 设计建议

如果你要继续把这个框架往研究或工程底盘推进，优先建议增强：

- 更强的 RAG ingestion / chunking / rerank
- 更强的 code interpreter，支持文件输入输出
- 更强的 supervisor 选择策略
- 更丰富的多智能体协作协议
- 更细粒度 observability 与 tracing
- 更强的 structured output 与自动修复


## 20. 实验排错清单

如果再次遇到问题，优先按这个顺序检查：

1. notebook 里是否误用了 `run_sync()`
2. 是否修改过本地代码但没有重启 kernel
3. 是否复用了旧的 `thread_id`
4. `.env` 是否被正确读取
5. `base_url` 是否被规整成 `/v1`
6. sandbox 的 `allowed_paths` 和 `command_allowlist` 是否允许当前执行
7. 多 agent 实验前是否先把 specialist agent 注册到 `AgentRegistry`
8. RAG 实验是否打开了 `RuntimeConfig(enable_retrieval=True)`

最常见的两条仍然是：

- notebook 要用 `await agent.run(...)`
- tool / multi-agent 调试失败时，请换新的 `thread_id` 或先 `clear_thread()`
